In [ ]:
# Celula unica - Importacao de dados GPKG para PostgreSQL
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============
# CONFIGURACOES 
# =============

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")

# Auto-detectar arquivos
CAR_ARQUIVOS = [f.name for f in PASTA_DADOS.glob("*car*.gpkg")]
MODULOS_FISCAIS_ARQUIVOS = [f.name for f in PASTA_DADOS.glob("*modulosfiscais*.gpkg")]

# Arquivo SIGEF (manual)
SIGEF_ARQUIVO = "pa_br_sigef_privado_20260107.gpkg"

# Controle de importacao
IMPORTAR_CAR = True
IMPORTAR_MODULOS = True
IMPORTAR_SIGEF = False  # Ativar apenas se necessário

# Controle geral
RECRIAR_TABELAS = False  # TRUE = sempre sobrescreve, FALSE = mantém existente

# Credenciais do banco
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_NAME = "geotec"

# ==========================================
# FUNCOES
# ==========================================

def conectar_banco():
    return create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

def criar_schemas(engine):
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS car"))
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS incra"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print("[OK] Schemas prontos")

def verificar_tabela(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except:
        return False

def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return result.fetchone()[0]
    except:
        return 0

def importar_gpkg(engine, caminho_arquivo, schema, tabela):
    print(f"  Lendo: {caminho_arquivo.name}")
    gdf = gpd.read_file(str(caminho_arquivo))
    print(f"  Registros: {len(gdf)}")

    # Se tiver geometria → PostGIS
    if 'geometry' in gdf.columns:
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4674)
        else:
            gdf = gdf.to_crs(epsg=4674)

        gdf.to_postgis(
            name=tabela,
            con=engine,
            schema=schema,
            if_exists='replace',
            index=False
        )
    else:
        df = pd.DataFrame(gdf)
        df.to_sql(
            name=tabela,
            con=engine,
            schema=schema,
            if_exists='replace',
            index=False
        )

    print(f"  OK → {schema}.{tabela}")
    return len(gdf)

# ==========================================
# MAIN
# ==========================================

def main():
    print("=" * 50)
    print("IMPORTACAO DE DADOS GEOPROCESSAMENTO")
    print("=" * 50)

    if not PASTA_DADOS.exists():
        print(f"[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return

    engine = conectar_banco()
    print("[OK] Conectado ao PostgreSQL")

    criar_schemas(engine)

    # =====================
    # CAR
    # =====================
    if IMPORTAR_CAR and CAR_ARQUIVOS:
        print("\n[1] Importando CAR...")

        for arquivo in CAR_ARQUIVOS:
            caminho = PASTA_DADOS / arquivo
            tabela = Path(arquivo).stem

            print(f"\n--> {arquivo}")

            if verificar_tabela(engine, "car", tabela) and not RECRIAR_TABELAS:
                print("  Mantido (já existe)")
            else:
                importar_gpkg(engine, caminho, "car", tabela)

    else:
        print("\n[1] CAR desativado ou nao encontrado")

    # =====================
    # MODULOS FISCAIS
    # =====================
    if IMPORTAR_MODULOS and MODULOS_FISCAIS_ARQUIVOS:
        print("\n[2] Importando Modulos Fiscais...")

        for arquivo in MODULOS_FISCAIS_ARQUIVOS:
            caminho = PASTA_DADOS / arquivo
            tabela = Path(arquivo).stem

            print(f"\n--> {arquivo}")

            if verificar_tabela(engine, "incra", tabela) and not RECRIAR_TABELAS:
                print("  Mantido (já existe)")
            else:
                importar_gpkg(engine, caminho, "incra", tabela)

    else:
        print("\n[2] Modulos Fiscais desativado ou nao encontrado")

    # =====================
    # SIGEF
    # =====================
    if IMPORTAR_SIGEF:
        print("\n[3] Importando SIGEF...")

        caminho = PASTA_DADOS / SIGEF_ARQUIVO
        tabela = Path(SIGEF_ARQUIVO).stem

        if verificar_tabela(engine, "incra", tabela) and not RECRIAR_TABELAS:
            print("  Mantido (já existe)")
        else:
            importar_gpkg(engine, caminho, "incra", tabela)

    else:
        print("\n[3] SIGEF desativado")

    # =====================
    # RESUMO
    # =====================
    print("\nResumo:")

    for arquivo in CAR_ARQUIVOS:
        tabela = Path(arquivo).stem
        print(f"  car.{tabela}: {contar_registros(engine, 'car', tabela)}")

    for arquivo in MODULOS_FISCAIS_ARQUIVOS:
        tabela = Path(arquivo).stem
        print(f"  incra.{tabela}: {contar_registros(engine, 'incra', tabela)}")

    if IMPORTAR_SIGEF:
        tabela = Path(SIGEF_ARQUIVO).stem
        print(f"  incra.{tabela}: {contar_registros(engine, 'incra', tabela)}")

    engine.dispose()

    print("\n" + "=" * 50)
    print("PRONTO! BASES IMPORTADAS COM SUCESSO")
    print("=" * 50)

if __name__ == "__main__":
    main()